# Colab SFT + GRPO Notebook for Jira Outlook Env
This notebook is a Colab-friendly version of the end-to-end SFT and GRPO workflow. It clones the repo, installs dependencies, runs SFT, then runs GRPO post-training from the SFT checkpoint, while saving plots inline and to PNG files.

## How to use
1. Open this notebook in Google Colab.
2. Enable a GPU runtime.
3. Set `REPO_URL` and `REPO_BRANCH` if you want a different branch.
4. Run the cells top to bottom.

In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get('JIRA_OUTLOOK_REPO_URL')
REPO_BRANCH = os.environ.get('JIRA_OUTLOOK_REPO_BRANCH')
WORKDIR = Path('/content/finals')
WORKDIR

In [ ]:
import os
if Path('/content').exists():
    !rm -rf /content/finals
    !git clone --branch "$REPO_BRANCH" "$REPO_URL" /content/finals
else:
    print('Not running in Colab /content environment; skipping clone.')

In [ ]:
import os
import sys
from pathlib import Path

WORKDIR = Path('/content/finals')
ROOT = WORKDIR if WORKDIR.exists() else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('repo_root =', ROOT)
print('python =', sys.executable)

In [ ]:
# Colab-friendly dependency installation
!python -m pip install --upgrade pip
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0
!pip install transformers>=4.56.0 datasets>=4.8.0 trl==1.2.0 peft>=0.19.0 accelerate>=1.13.0 matplotlib>=3.10.0 openai>=1.51.0 pydantic>=2.0.0 jupyter nbconvert nbformat

In [ ]:
import json
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer, GRPOConfig, GRPOTrainer

print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))

In [ ]:
from training.prepare_sft_data import main as prepare_sft_data

prepare_sft_data()

In [ ]:
dataset_path = ROOT / 'data' / 'training' / 'sft_train.jsonl'
dataset = load_dataset('json', data_files=str(dataset_path), split='train')
print(dataset)
print(dataset[0])

In [ ]:
def format_example(example):
    parts = []
    for msg in example['messages']:
        parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>")
    return {'text': '\n'.join(parts)}

formatted = dataset.map(format_example)
print(formatted[0]['text'][:800])

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'
OUTPUT_DIR = ROOT / 'training' / 'output' / 'qwen3_4b_sft_colab'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype='auto',
    device_map='auto',
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'gate_proj'],
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3.0,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    logging_steps=1,
    save_steps=100,
    bf16=torch.cuda.is_available(),
    report_to='none',
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted,
    processing_class=tokenizer,
    peft_config=peft_config,
    formatting_func=lambda ex: ex['text'],
)

In [ ]:
train_result = trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(train_result)

In [ ]:
state_candidates = sorted(OUTPUT_DIR.glob('checkpoint-*/trainer_state.json'))
state_path = state_candidates[-1] if state_candidates else OUTPUT_DIR / 'trainer_state.json'
state = json.loads(state_path.read_text())
logs = [entry for entry in state['log_history'] if 'loss' in entry]
steps = [entry['step'] for entry in logs]
losses = [entry['loss'] for entry in logs]
accs = [entry.get('mean_token_accuracy') for entry in logs]

plot_dir = ROOT / 'training' / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(8, 5))
plt.plot(steps, losses, marker='o')
plt.title('SFT Training Loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(True)
loss_png = plot_dir / 'sft_loss_curve_colab.png'
plt.savefig(loss_png, dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(steps, accs, marker='o')
plt.title('SFT Mean Token Accuracy')
plt.xlabel('Step')
plt.ylabel('Accuracy')
plt.grid(True)
acc_png = plot_dir / 'sft_accuracy_curve_colab.png'
plt.savefig(acc_png, dpi=150, bbox_inches='tight')
plt.show()

print('Saved:', loss_png)
print('Saved:', acc_png)

## Notes
- This notebook keeps the existing local notebook unchanged.
- It assumes the repository contains `data/`, `training/`, and `notebooks/` in the checked-out branch.
- For private GitHub Enterprise repos in Colab, you may need to authenticate before cloning.

## GRPO Post-Training
This section starts from the SFT checkpoint produced above and runs GRPO-based RL post-training. The plots are shown inline and also saved as PNG files.

In [ ]:
grpo_source = load_dataset("json", data_files=str(ROOT / "data" / "training" / "sft_train.jsonl"), split="train")

def to_grpo_prompt(example):
    user_msg = next(msg["content"] for msg in example["messages"] if msg["role"] == "user")
    assistant_msg = next(msg["content"] for msg in example["messages"] if msg["role"] == "assistant")
    prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n"
    return {"prompt": prompt[:6000], "reference_action": assistant_msg}

grpo_dataset = grpo_source.map(to_grpo_prompt, remove_columns=grpo_source.column_names)
print(grpo_dataset)
print(grpo_dataset[0])

In [ ]:
grpo_reward_history = []

def parse_json_action(text):
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(text[start:end + 1])
    except Exception:
        return None

def structural_reward(completions, reference_action, **kwargs):
    rewards = []
    for completion, reference in zip(completions, reference_action):
        payload = parse_json_action(completion)
        ref_payload = parse_json_action(reference)
        reward = -0.2
        if payload is not None:
            reward = 0.1
            if ref_payload is not None and payload.get("tool") == ref_payload.get("tool"):
                reward += 0.35
            if ref_payload is not None and payload.get("resolution") == ref_payload.get("resolution"):
                reward += 0.35
            if ref_payload is not None and payload.get("resolution_notes") == ref_payload.get("resolution_notes"):
                reward += 0.2
        grpo_reward_history.append(float(reward))
        rewards.append(float(reward))
    return rewards

def extract_grpo_losses(log_history):
    losses = []
    for entry in log_history:
        if isinstance(entry, dict):
            if "loss" in entry:
                losses.append(float(entry["loss"]))
            elif "train_loss" in entry:
                losses.append(float(entry["train_loss"]))
    return losses

In [ ]:
GRPO_OUTPUT_DIR = ROOT / "training" / "output" / "qwen3_4b_grpo_colab"

grpo_config = GRPOConfig(
    output_dir=str(GRPO_OUTPUT_DIR),
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=128,
    logging_steps=1,
    num_train_epochs=2,
    save_steps=3,
    report_to="none",
    bf16=torch.cuda.is_available(),
)

base_for_grpo = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto", device_map="auto")
sft_loaded = PeftModel.from_pretrained(base_for_grpo, str(OUTPUT_DIR))
merged_for_grpo = sft_loaded.merge_and_unload()

grpo_peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
)

grpo_tokenizer = AutoTokenizer.from_pretrained(str(OUTPUT_DIR), use_fast=True)
if grpo_tokenizer.pad_token is None:
    grpo_tokenizer.pad_token = grpo_tokenizer.eos_token

grpo_trainer = GRPOTrainer(
    model=merged_for_grpo,
    reward_funcs=structural_reward,
    args=grpo_config,
    train_dataset=grpo_dataset.select(range(min(12, len(grpo_dataset)))),
    processing_class=grpo_tokenizer,
    peft_config=grpo_peft_config,
)

grpo_train_result = grpo_trainer.train()
grpo_trainer.save_model(str(GRPO_OUTPUT_DIR))
print(grpo_train_result)

In [ ]:
checkpoint_states = sorted((GRPO_OUTPUT_DIR).glob("checkpoint-*/trainer_state.json"))
trainer_state_path = checkpoint_states[-1] if checkpoint_states else GRPO_OUTPUT_DIR / "trainer_state.json"
trainer_state = json.loads(trainer_state_path.read_text()) if trainer_state_path.exists() else {"log_history": []}
log_history = trainer_state.get("log_history", [])
grop_losses = extract_grpo_losses(log_history)
reward_means = [float(entry["reward"]) for entry in log_history if isinstance(entry, dict) and "reward" in entry]
steps = [int(entry["step"]) for entry in log_history if isinstance(entry, dict) and "step" in entry and "reward" in entry]

plot_dir = ROOT / "training" / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)
reward_png = plot_dir / "grpo_reward_curve_colab.png"
loss_png = plot_dir / "grpo_loss_curve_colab.png"

plt.figure(figsize=(8, 5))
plt.plot(steps if steps else range(1, len(reward_means) + 1), reward_means, marker="o")
plt.title("GRPO Mean Reward by Step")
plt.xlabel("Training step")
plt.ylabel("Mean reward")
plt.grid(True)
plt.savefig(reward_png, dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(grop_losses) + 1), grop_losses, marker="o")
plt.title("GRPO Loss by Step")
plt.xlabel("Logged step")
plt.ylabel("Loss")
plt.grid(True)
plt.savefig(loss_png, dpi=150, bbox_inches="tight")
plt.show()

print("Saved:", reward_png)
print("Saved:", loss_png)
print("reward_points", len(reward_means))
print("loss_points", len(grop_losses))

## GRPO Notes
- This Colab notebook now supports both SFT and GRPO post-training.
- The GRPO section starts from the SFT checkpoint produced earlier in the notebook.
- Reward and loss curves are displayed inline and saved to PNG files for later inspection.